In [1]:
# Block 1: Load Fact Tables
import pandas as pd
import numpy as np

print("Loading Fact Tables...\n")

df_att = pd.read_csv("D:\HR Project\HR data\Raw\Fact_Attendance.csv")
df_comp = pd.read_csv("D:\HR Project\HR data\Raw\Fact_Compensation.csv")
df_eng = pd.read_csv("D:\HR Project\HR data\Raw\Fact_Engagement.csv")
df_perf = pd.read_csv("D:\HR Project\HR data\Raw\Fact_Performance.csv")

print(f"Fact_Attendance loaded: {df_att.shape[0]:,} rows.")
print(f"Fact_Compensation loaded: {df_comp.shape[0]:,} rows.")
print(f"Fact_Engagement loaded: {df_eng.shape[0]:,} rows.")
print(f"Fact_Performance loaded: {df_perf.shape[0]:,} rows.")

Loading Fact Tables...

Fact_Attendance loaded: 725,408 rows.
Fact_Compensation loaded: 67,769 rows.
Fact_Engagement loaded: 67,566 rows.
Fact_Performance loaded: 67,904 rows.


In [2]:
# Block 2: Clean Fact_Attendance
print("Cleaning Fact_Attendance...")

# 1. Drop exact duplicates
initial_rows = len(df_att)
df_att.drop_duplicates(inplace=True)
print(f"   -> Removed {initial_rows - len(df_att):,} duplicated rows.")

# 2. Standardize Business Travel Text
travel_map = {'low': 'Low', 'Med': 'Medium', 'HIGH': 'High'}
df_att['business_travel_frequency'] = df_att['business_travel_frequency'].replace(travel_map)

# 3. Create Flags for logical errors
df_att['negative_sick_leave_flag'] = df_att['sick_leaves_taken'] < 0
df_att['high_overtime_flag'] = df_att['overtime_hours'] > 100
df_att['high_working_hours_flag'] = df_att['total_working_hours'] > 300

# 4. Fix absolute logic errors (Convert negative sick leaves to positive)
df_att['sick_leaves_taken'] = df_att['sick_leaves_taken'].abs()

print("Fact_Attendance cleaned and flagged.")

Cleaning Fact_Attendance...
   -> Removed 5,757 duplicated rows.
Fact_Attendance cleaned and flagged.


In [3]:
# Block 3: Clean Fact_Compensation
print("\n Cleaning Fact_Compensation...")

# 1. Drop exact duplicates
initial_rows = len(df_comp)
df_comp.drop_duplicates(inplace=True)
print(f"   -> Removed {initial_rows - len(df_comp):,} duplicated rows.")

# 2. Convert Dates
df_comp['effective_date'] = pd.to_datetime(df_comp['effective_date'], errors='coerce')

# 3. Create Flags
df_comp['negative_bonus_flag'] = df_comp['annual_bonus_egp'] < 0
# Max director salary is 250k, anything way above is an outlier (like the 1.3M we found)
df_comp['salary_outlier_flag'] = df_comp['monthly_salary_egp'] > 250000 

# 4. Fixes
df_comp['annual_bonus_egp'] = df_comp['annual_bonus_egp'].abs() # Fix negative bonuses

print("Fact_Compensation cleaned and flagged.")


 Cleaning Fact_Compensation...
   -> Removed 203 duplicated rows.
Fact_Compensation cleaned and flagged.


In [4]:
# Block 4: Clean Fact_Engagement
print("\n Cleaning Fact_Engagement...")

df_eng.drop_duplicates(inplace=True)
df_eng['survey_date'] = pd.to_datetime(df_eng['survey_date'], errors='coerce')

# Fix WLB score = 0 (scale is 1-5)
df_eng['invalid_wlb_flag'] = df_eng['work_life_balance_score'] == 0
# Convert 0s to NaN so they don't drag the average down incorrectly in Tableau
df_eng.loc[df_eng['work_life_balance_score'] == 0, 'work_life_balance_score'] = np.nan

print(f"   -> Fixed {df_eng['invalid_wlb_flag'].sum()} invalid WLB scores (0 -> NaN).")
print("Fact_Engagement cleaned and flagged.")


 Cleaning Fact_Engagement...
   -> Fixed 68 invalid WLB scores (0 -> NaN).
Fact_Engagement cleaned and flagged.


In [5]:
# Block 5: Clean Fact_Performance
print("\n Cleaning Fact_Performance...")

# 1. Drop exact duplicates
initial_rows = len(df_perf)
df_perf.drop_duplicates(inplace=True)
print(f"   -> Removed {initial_rows - len(df_perf):,} duplicated rows.")

df_perf['review_date'] = pd.to_datetime(df_perf['review_date'], errors='coerce')

# 2. Flag and fix invalid performance scores
df_perf['invalid_score_flag'] = ~df_perf['performance_score'].isin([1, 2, 3, 4, 5])

# Replace 0 and 6 with NaN (or we can leave as is, but NaN is safer for averages)
df_perf.loc[df_perf['invalid_score_flag'], 'performance_score'] = np.nan

print(f"   -> Fixed {df_perf['invalid_score_flag'].sum()} invalid performance scores.")
print("Fact_Performance cleaned and flagged.")


 Cleaning Fact_Performance...
   -> Removed 338 duplicated rows.
   -> Fixed 136 invalid performance scores.
Fact_Performance cleaned and flagged.


In [6]:
# Block 6: Save Cleaned Facts
print("\n Saving cleaned fact tables to CSV...")

df_att.to_csv("Fact_Attendance_Cleaned.csv", index=False)
df_comp.to_csv("Fact_Compensation_Cleaned.csv", index=False)
df_eng.to_csv("Fact_Engagement_Cleaned.csv", index=False)
df_perf.to_csv("Fact_Performance_Cleaned.csv", index=False)

print(" All Fact Tables saved successfully with '_Cleaned.csv' suffix.")


 Saving cleaned fact tables to CSV...
 All Fact Tables saved successfully with '_Cleaned.csv' suffix.
